In [5]:
import pandas as pd
df=pd.read_csv(r'/adfJuly2025final_1.csv')
df.shape

(43, 36)

In [6]:
df.head()

,DIST,Basic Procurement Govt,Basic Procurement Co.op,Basic Procurement Pvt,Total Procurement,Interstate Recipet Govt,Interstate Recipet Co.op,Interstate Recipet Pvt,Interstate Recipet Total,Outerstate Receipt Govt,...,Govt Bi-Products,Co.op Bi-Products,Pvt Bi-Products,Total Bi-Products,Govt Conversion,Co.op Conversion,Pvt Conversion,Total Conversion,Mahanand,Co.Op to Mahanand
0,Pune,0,5.80,23.62,29.41,0,0.00,57.30,57.30,0,...,0,0.04,26.64,26.68,0,0.00,34.80,34.80,0,0.45
1,Sangli,0,4.17,10.67,14.84,0,0.15,0.00,0.15,0,...,0,0.15,0.51,0.65,0,0.64,1.95,2.59,0,0.05
2,Satara,0,0.97,15.34,16.31,0,0.15,3.82,3.97,0,...,0,0.04,2.20,2.24,0,0.00,3.82,3.82,0,0.03
3,Solapur,0,1.61,12.70,14.31,0,0.75,0.00,0.75,0,...,0,0.16,0.03,0.19,0,2.87,0.00,2.87,0,0.00
4,Kolhapr,0,14.12,0.79,14.91,0,2.17,0.89,3.06,0,...,0,0.33,0.45,0.78,0,0.87,0.70,1.57,0,0.00


In [7]:
df.columns = [c.replace('\x81', '') for c in df.columns]
df.columns.tolist()

['DIST',
 'Basic Procurement Govt',
 'Basic Procurement Co.op',
 'Basic Procurement Pvt',
 'Total Procurement',
 'Interstate Recipet Govt',
 'Interstate Recipet Co.op',
 'Interstate Recipet Pvt',
 'Interstate Recipet Total',
 'Outerstate Receipt Govt',
 'Outerstate Receipt Co.op',
 'Outerstate Receipt Pvt',
 'Outerstate Receipt Total ',
 'Total Procurement & Receipt',
 'Govt Pouch Sale',
 'Co.op Pouch Sale',
 'Pvt Pouch Sale',
 'Total Pouch Sale',
 'Govt Retail Sale',
 'Co.op Retail Sale',
 'Pvt Retail Sale',
 'Total Retail Sale',
 'Govt Outerstate Sale',
 'Co.op Outerstate Sale',
 'Pvt Outerstate Sale',
 'Total Outerstate Sale',
 'Govt Bi-Products',
 'Co.op Bi-Products',
 'Pvt Bi-Products',
 'Total Bi-Products',
 'Govt Conversion',
 'Co.op Conversion',
 'Pvt Conversion',
 'Total Conversion',
 'Mahanand',
 'Co.Op to Mahanand']

In [8]:
#which rows are non-district summary rows
df['DIST'].tolist()

['Pune',
 'Sangli',
 'Satara',
 'Solapur',
 'Kolhapr',
 'Total',
 'A.Nagar',
 'Nasik',
 'Jalgaon',
 'Dhule',
 'Nandrbr',
 'Total',
 'A"bad',
 'Jalna',
 'Beed',
 'Parbhani',
 'Hingoli',
 'Nanded',
 'Dharashiv',
 'Latur',
 'Total',
 'Thane',
 'Raigad',
 'Ratnagiri',
 'Sindrg.',
 'Total',
 'Amarvti',
 'Akola',
 'Washim',
 'Buldhna',
 'Yvtmal',
 'Total',
 'Nagpr',
 'Wardha',
 'Bhandra',
 'Gondia',
 'Chandrpur',
 'Gdchrli',
 'Total',
 'DIST TTL',
 'Mumbai',
 'MHND',
 'Grand Total']

In [43]:
# Define which values are NOT actual districts
summary_labels = ['Total', 'DIST TTL', 'Mumbai', 'MHND', 'Grand Total']
df_summary = df[df['DIST'].isin(summary_labels)].copy()
df_districts = df[~df['DIST'].isin(summary_labels)].copy()

total_rows_idx = df_summary[df_summary['DIST'] == 'Total'].index
print(len(total_rows_idx))

6


In [44]:
district_name_fix = {
    'A.Nagar': 'Ahmednagar',
    'Nandrbr': 'Nandurbar',
    'A"bad': 'Aurangabad',
    'Sindrg.': 'Sindhudurg',
    'Nagpr': 'Nagpur',
    'Gdchrli': 'Gadchiroli',
    'Kolhapr': 'Kolhapur',
    'Yvtmal': 'Yavatmal',
    'Amarvti': 'Amravati',
    'Bhandra': 'Bhandara'
}
df_districts['DIST'] = df_districts['DIST'].replace(district_name_fix)

In [45]:
# Check for duplicate districts
df_districts['DIST'].duplicated().sum()

np.int64(0)

In [46]:
# Check for negative values
numeric_cols = df_districts.select_dtypes(include='number').columns
(df_districts[numeric_cols] < 0).sum().sum()

# Check does Total Procurement roughly match sum of Govt+Coop+Pvt columns?
df_districts['calc_total_procurement'] = (
    df_districts['Basic Procurement Govt'] +
    df_districts['Basic Procurement Co.op'] +
    df_districts['Basic Procurement Pvt']
)
df_districts[['DIST','Total Procurement','calc_total_procurement']].head(10)

,DIST,Total Procurement,calc_total_procurement
0,Pune,29.41,29.42
1,Sangli,14.84,14.84
2,Satara,16.31,16.31
3,Solapur,14.31,14.31
4,Kolhapur,14.91,14.91
6,Ahmednagar,37.30,37.30
7,Nasik,7.41,7.41
8,Jalgaon,3.04,3.04
9,Dhule,1.35,1.35
10,Nandurbar,0.00,0.00


In [47]:
#Melting the data
sector_cols = ['Basic Procurement Govt', 'Basic Procurement Co.op', 'Basic Procurement Pvt']

df_long = df_districts.melt(
    id_vars='DIST',
    value_vars=sector_cols,
    var_name='Sector',
    value_name='Procurement_Lakhs'
)
df_long['Sector'] = df_long['Sector'].str.replace('Basic Procurement ', '')
df_long.tail(10)

,DIST,Sector,Procurement_Lakhs
89,Akola,Pvt,0.40
90,Washim,Pvt,0.45
91,Buldhna,Pvt,1.24
92,Yavatmal,Pvt,1.13
93,Nagpur,Pvt,1.89
94,Wardha,Pvt,1.01
95,Bhandara,Pvt,1.50
96,Gondia,Pvt,0.37
97,Chandrpur,Pvt,0.21
98,Gadchiroli,Pvt,0.01


In [48]:
#Self-Sufficiency Ratio
df_districts['Self_Sufficiency_Ratio'] = (
    df_districts['Total Procurement'] / df_districts['Total Procurement & Receipt']
).round(3)
df_districts[['DIST', 'Total Procurement', 'Total Procurement & Receipt','Self_Sufficiency_Ratio']].head()

,DIST,Total Procurement,Total Procurement & Receipt,Self_Sufficiency_Ratio
0,Pune,29.41,87.39,0.337
1,Sangli,14.84,15.40,0.964
2,Satara,16.31,20.27,0.805
3,Solapur,14.31,16.60,0.862
4,Kolhapur,14.91,20.24,0.737


In [49]:
#Procurement-to-sale ratio per district
df_districts['Total Sale'] = (
    df_districts['Total Pouch Sale'] +
    df_districts['Total Retail Sale'] +
    df_districts['Total Outerstate Sale'])

df_districts['Procurement_to_Sale_Ratio'] = (
    (df_districts['Total Sale'] + df_districts['Total Bi-Products'] + df_districts['Total Conversion'])
    / df_districts['Total Procurement & Receipt']
).round(3)
df_districts[['DIST', 'Procurement_to_Sale_Ratio']].head()

,DIST,Procurement_to_Sale_Ratio
0,Pune,0.994
1,Sangli,0.997
2,Satara,1.002
3,Solapur,0.992
4,Kolhapur,1.059


In [50]:
# Flag low-volume districts where ratios are statistically noisy
df_districts['Volume_Tier'] = pd.cut(
    df_districts['Total Procurement & Receipt'],
    bins=[-0.01, 1, 10, 1000],
    labels=['Low Volume (<1 lakh)', 'Mid Volume (1-10 lakh)', 'High Volume (10+ lakh)']
)

df_districts[['DIST', 'Total Procurement & Receipt', 'Self_Sufficiency_Ratio', 'Volume_Tier']].sort_values('Total Procurement & Receipt')

,DIST,Total Procurement & Receipt,Self_Sufficiency_Ratio,Volume_Tier
10,Nandurbar,0.00,NaN,Low Volume (<1 lakh)
37,Gadchiroli,0.03,1.000,Low Volume (<1 lakh)
15,Parbhani,0.16,0.938,Low Volume (<1 lakh)
36,Chandrpur,0.21,1.000,Low Volume (<1 lakh)
24,Sindhudurg,0.27,0.593,Low Volume (<1 lakh)
16,Hingoli,0.31,0.774,Low Volume (<1 lakh)
26,Amravati,0.37,1.000,Low Volume (<1 lakh)
35,Gondia,0.37,1.000,Low Volume (<1 lakh)
27,Akola,0.40,1.000,Low Volume (<1 lakh)
23,Ratnagiri,0.41,1.000,Low Volume (<1 lakh)


In [51]:
# Cross-Validation
python_total = df_districts['Total Procurement & Receipt'].sum()
govt_grand_total = df_summary.loc[df_summary['DIST'] == 'Grand Total', 'Total Procurement & Receipt'].values[0]

print(f"My calculated total: {python_total}")
print(f"Government's Grand Total: {govt_grand_total}")

My calculated total: 282.48999999999995
Government's Grand Total: 283.69


In [52]:
df_summary[['DIST', 'Total Procurement & Receipt']]

,DIST,Total Procurement & Receipt
5,Total,159.91
11,Total,55.99
20,Total,16.64
25,Total,36.00
31,Total,3.61
38,Total,10.34
39,DIST TTL,282.49
40,Mumbai,0.00
41,MHND,1.21
42,Grand Total,283.69


In [53]:
print("Original rows:", len(df))
print("District rows:", len(df_districts))
print("Summary rows:", len(df_summary))
print("Check: district + summary should equal original:", len(df_districts) + len(df_summary) == len(df))

Original rows: 43
District rows: 33
Summary rows: 10
Check: district + summary should equal original: True


In [54]:
division_map = {
    'Pune': 'Pune', 'Sangli': 'Pune', 'Satara': 'Pune', 'Solapur': 'Pune', 'Kolhapur': 'Pune',
    'Ahmednagar': 'Nashik', 'Nasik': 'Nashik', 'Jalgaon': 'Nashik', 'Dhule': 'Nashik', 'Nandurbar': 'Nashik',
    'Aurangabad': 'Aurangabad', 'Jalna': 'Aurangabad', 'Beed': 'Aurangabad', 'Parbhani': 'Aurangabad',
    'Hingoli': 'Aurangabad', 'Nanded': 'Aurangabad', 'Dharashiv': 'Aurangabad', 'Latur': 'Aurangabad',
    'Thane': 'Konkan', 'Raigad': 'Konkan', 'Ratnagiri': 'Konkan', 'Sindhudurg': 'Konkan',
    'Amravati': 'Amravati', 'Akola': 'Amravati', 'Washim': 'Amravati', 'Buldhna': 'Amravati', 'Yavatmal': 'Amravati',
    'Nagpur': 'Nagpur', 'Wardha': 'Nagpur', 'Bhandara': 'Nagpur', 'Gondia': 'Nagpur', 'Chandrpur': 'Nagpur', 'Gadchiroli': 'Nagpur'
}

df_districts['Division'] = df_districts['DIST'].map(division_map)
df_districts[['DIST', 'Division']].head(10)

,DIST,Division
0,Pune,Pune
1,Sangli,Pune
2,Satara,Pune
3,Solapur,Pune
4,Kolhapur,Pune
6,Ahmednagar,Nashik
7,Nasik,Nashik
8,Jalgaon,Nashik
9,Dhule,Nashik
10,Nandurbar,Nashik


In [57]:
division_order = ['Pune Total', 'Nashik Total', 'Aurangabad Total', 'Konkan Total', 'Amravati Total', 'Nagpur Total']
df_summary.loc[total_rows_idx, 'DIST'] = division_order

In [58]:
print(len(total_rows_idx))
print(df_summary['DIST'].tolist())

6
['Pune Total', 'Nashik Total', 'Aurangabad Total', 'Konkan Total', 'Amravati Total', 'Nagpur Total', 'DIST TTL', 'Mumbai', 'MHND', 'Grand Total']


In [59]:
#Exporting to Excel
with pd.ExcelWriter('maharashtra_dairy_cleaned.xlsx', engine='openpyxl') as writer:
    df_districts.to_excel(writer, sheet_name='District_Data', index=False)
    df_long.to_excel(writer, sheet_name='Sector_Long_Format', index=False)
    df_summary.to_excel(writer, sheet_name='Govt_Summary_Rows', index=False)

from google.colab import files
files.download('maharashtra_dairy_cleaned.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>